In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [3]:
from langchain_community.document_loaders import PyPDFLoader, MergedDataLoader

In [4]:
docs = MergedDataLoader([
    PyPDFLoader("data/GTB_gold_Nov23.pdf"),
    PyPDFLoader("data/GTB_platinum_Nov23.pdf")
]).load()

In [5]:
len(docs)

53

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

slices = splitter.split_documents(docs)

In [8]:
len(slices)

200

In [9]:
from langchain_openai.embeddings import OpenAIEmbeddings

In [10]:
embedding_model = OpenAIEmbeddings()

embedding_model.model

'text-embedding-ada-002'

In [11]:
from langchain_community.vectorstores import InMemoryVectorStore

In [12]:
vectorstore = InMemoryVectorStore.from_documents(
    documents=slices,
    embedding=embedding_model
)

In [23]:
retriever = vectorstore.as_retriever()

In [18]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

In [40]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """Gere cinco consultas de pesquisa para um banco de dados vetorial a partir
     de uma pergunta do usuario, permitindo uma resposta mais precisa para a busca semantica

     Gere somente o texto das consultas separadas por quebra de linha e nada de texto adicional
     """),
    ("human", "{question}")
])

query_model = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.5,
    api_key=openai_api_key
)

rewrite_chain = rewrite_prompt | query_model | CommaSeparatedListOutputParser()

In [41]:
query = "Como lidar com cartão roubado?"

In [42]:
rewrite_chain.invoke(query)

['cartão roubado procedimento bloqueio cancelamento denúncia fraude segurança financeira  ',
 'como agir cartão clonado passos prevenção segurança financeira  ',
 'medidas para cartão roubado ou perdido orientações segurança bancária  ',
 'o que fazer ao perder cartão bancário procedimentos recuperação proteção  ',
 'dicas para evitar roubo de cartão dicas segurança financeira prevenção']

In [43]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [44]:
multi_query_retriever = MultiQueryRetriever(
    retriever=retriever,
    llm_chain=rewrite_chain
)

In [45]:
multi_query_retriever.invoke(query)

[Document(id='b2b22bc6-f30c-4cd0-af60-b3bb2c599c7f', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-26T10:05:20-03:00', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_enabled': 'true', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_setdate': '2023-06-21T14:24:39Z', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_method': 'Privileged', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_name': 'Restricted', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_siteid': 'f06fa858-824b-4a85-aacb-f372cfdc282e', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_actionid': '8050f60f-7cb7-4b1b-99e0-7f18410feb72', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_contentbits': '0', 'title': 'GTB WE', 'author': 'Zachary A. Cardoza', 'moddate': '2023-10-26T10:05:20-03:00', 'source': 'data/GTB_gold_Nov23.pdf', 'total_pages': 24, 'page': 17, 'page_label': '18'}, page_content='* É possível que seja solicitado ao Portado

In [47]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [48]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Responda utilizando exclusivamente o conteúdo fornecido: \n\nContexto: \n{contexto}"),
    ("human", "{query}")
])

In [49]:
from langchain_core.runnables import RunnablePassthrough

In [51]:
multi_query_rag_chain = (
    {
        "contexto": RunnablePassthrough() | multi_query_retriever | format_docs,
        "query": RunnablePassthrough()
    } | rag_prompt | query_model | StrOutputParser()
)

In [56]:
multi_query_rag_chain.invoke(query)

'Se seu cartão foi roubado, você deve entrar em contato com o Mastercard Global Service para relatar o incidente. Caso não seja possível acessar o número gratuito, ligue para o Mastercard® Global Service no número 1-636-722-8881 (Português) ou acesse o site www.mycardbenefits.com. É importante comunicar o roubo imediatamente para que as providências necessárias sejam tomadas, incluindo a proteção de sua conta e a possível investigação do incidente.'